# Revision 2 — deployment runtime and controlled CPU timing

Addresses reviewer 1 comments 6 and 7. Select **Runtime → Change runtime type → CPU (hardware accelerator: None)**, then **Run all**. Leave other heavy workloads idle during timing.

This notebook reads the existing Google Drive project directly and saves every result there. No archive upload is needed. It reloads the **twenty original proposed-student checkpoints** (ten seeds, two datasets), converts them to float32 TFLite, reuses both selected saved SVMs and creates equivalent compact coefficient exports for linear SVCs. **No teacher, coalition or student training occurs.** One auxiliary WISDM linear SVC is fitted on original training features only, using the best linear C from the original validation search; this is a separate baseline, not a replacement for the selected RBF model.

The stages are: frozen input checks and guard tests → all-model validation/conversion checks → separate short smoke → one complete CPU timing session. Stop on a failed check; do not relax tolerances. Original validation/test accuracy and macro-F1 must agree within 0.0001. TFLite and traced TensorFlow paths must agree with loaded Keras logits within atol=rtol=0.0001 **and predict exactly the same label for every validation and test window**. Compact linear SVC labels must match its support-vector form exactly.

Primary deployment timing uses optimized TFLite builtin CPU kernels with default delegates explicitly disabled, one thread and no quantization/Select TF/custom operators. Keras eager and traced TensorFlow paths are retained to characterize call-path differences. The research run has 65 conditions: 60 student/runtime conditions, selected UCI SVC and its compact equivalent, selected WISDM RBF SVC, and the separately identified WISDM linear SVC plus compact equivalent. Three timing rounds retain raw samples: 30 warm-ups + 300 batch-one calls; 5 warm-ups + 30 batch-128 calls per condition per round. Cases are deterministically shuffled between rounds. Timing uses matching fixed validation-window indices for raw-window and engineered-feature inputs.

Timing includes classifier calls, output materialization and argmax/voting. Inputs, tensors and interpreter allocations are prepared beforehand. Sensor processing, feature extraction, normalization, model loading, tracing and conversion are excluded. Artifact and numeric-state sizes are classifier-only; runtime binaries and preprocessing state are excluded. This is a controlled Colab CPU audit, not physical-edge-device, end-to-end or energy evidence. Retain all outcomes and separate across-model seed variation from within-session timing variation. No timing results exist until the final completion marker is saved.

Each Run-all creates a fresh session folder so timings from different hosts/runs cannot be mixed. If a runtime disconnects, reconnect and Run all to start a fresh session. Do not rename old results or reuse partial timings.


In [ ]:
# Mount persistent Drive storage and create a new, isolated session.
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
import uuid,json,sys,subprocess,os
PROJECT=Path('/content/drive/MyDrive/makd-har')
EXPERIMENT=PROJECT/'revision_2/runtime_benchmark'
assert (EXPERIMENT/'frozen-inputs.json').is_file(), 'Wait for Google Drive to synchronize the prepared experiment files.'
SESSION=PROJECT/'results/revision-2/runtime-benchmark'/('session-'+datetime.now(ZoneInfo('Asia/Tehran')).strftime('%Y%m%d-%H%M%S')+'-'+uuid.uuid4().hex[:8])
SESSION.mkdir(parents=True,exist_ok=False)
print('Inputs:',EXPERIMENT)
print('Persistent result session:',SESSION)


In [ ]:
# Require the repaired package before executing any phase.
assert json.loads((EXPERIMENT/'frozen-inputs.json').read_text())['version']=='revision2-runtime-benchmark-v2', 'Wait for Drive to synchronize the repaired package, then rerun this cell.'
# Saved SVCs require the original library version. All numerical work runs in fresh subprocesses.
subprocess.check_call([sys.executable,'-m','pip','install','--quiet','scikit-learn==1.6.1'])

def run_audit(*flags):
    label='verify' if '--verify-only' in flags else ('preflight' if '--prepare' in flags else ('smoke' if '--smoke' in flags else 'research'))
    log=SESSION/(label+'.log')
    command=[sys.executable,'-u',str(EXPERIMENT/'run_runtime.py'),'--drive-root',str(PROJECT),'--session',str(SESSION),*flags]
    print('Running:',label,'— full log:',log,flush=True)
    notices=0
    with log.open('w') as stream:
        with subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,env=dict(os.environ,TF_CPP_MIN_LOG_LEVEL='2')) as process:
            for line in process.stdout:
                stream.write(line);stream.flush()
                if 'NodeDef mentions attribute use_unbounded_threadpool' in line:notices+=1
                else:print(line,end='',flush=True)
            code=process.wait()
    if notices:print(notices,'compatibility notices retained in the full log.')
    if code:raise RuntimeError(f'{label} stopped. Read the full error above or in {log}; do not run later phases.')

run_audit('--verify-only')
subprocess.check_call([sys.executable,str(EXPERIMENT/'test_runtime_guards.py'),'-v'])
print('Frozen input checks and eight guard tests passed.')


In [ ]:
# Convert all twenty saved students, check original validation metrics and SVM representations.
# One original-training-only auxiliary linear SVC fit; no test inference and no neural training.
run_audit('--prepare')
gate=json.loads((SESSION/'preflight-completed.json').read_text())
assert gate['status']=='complete' and gate['students']==20 and gate['svm_fits']==1 and gate['neural_training_fits']==0
assert gate['test_evaluated'] is False
print('ALL VALIDATION AND CONVERSION CHECKS PASSED.')


In [ ]:
# Separate short smoke: seed 11 on both datasets; excluded from research timings.
run_audit('--smoke')
smoke=json.loads((SESSION/'smoke-completed.json').read_text())
assert smoke['status']=='complete' and smoke['student_checkpoints']==2 and smoke['conditions']==11
print('SMOKE PASSED. Research timing can proceed.')


In [ ]:
# All twenty checkpoints and five SVM representations in one CPU process, three timing rounds.
run_audit()


In [ ]:
# A complete marker is written only after all parity checks and research timings finish.
completion=json.loads((SESSION/'completed.json').read_text())
assert completion['status']=='complete' and completion['mode']=='research'
assert completion['student_checkpoints']==20 and completion['conditions']==65 and completion['timing_records']==195
assert completion['neural_training_fits']==0 and completion['auxiliary_svm_fits']==1
print('COMPLETE: 20 student checkpoints, 65 runtime/model conditions, 195 timing records.')
print('Results saved on Google Drive:',SESSION)
print('Tell me when this finishes so I can independently verify and analyze the results in OneDrive.')
